In [ ]:
import pandas as pd
import torch
import gc
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

def generate_embeddings(model_name, df, task_description="Extract features for movie recommendation"):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"\nLoading model: {model_name}...")
    model = SentenceTransformer(model_name, trust_remote_code=True)
    if torch.cuda.is_available():
        model = model.to('cuda')
    if torch.mps.is_available():
        model = model.to('mps')

    print(f"Generating embeddings for {len(df)} rows individually...")
    all_embeddings = []

    # Process row by row as requested
    for _, row in tqdm(df.iterrows(), total=len(df)):
        query = f"""
Instruct: {task_description}
Title: {row['title']}
Summary: {row['plot_summary']}
        """.replace('\n', ' ')
        # Encode single query (returns a 2D array, we take the first element)
        embedding = model.encode([query], show_progress_bar=False, truncate_dim=1024)[0]
        all_embeddings.append(embedding)

    # Convert list of embeddings to DataFrame
    import numpy as np
    embeddings_array = np.array(all_embeddings)
    emb_df = pd.DataFrame(embeddings_array, columns=[f'embedding_{i}' for i in range(embeddings_array.shape[1])])

    # result structure: | movie_id | genres | moods | themes | keywords | embedding_0 | ... |
    res_cols = []
    if 'movie_id' in df.columns:
        res_cols.append(df[['movie_id']].reset_index(drop=True))

    for col in ['genres', 'moods', 'themes', 'keywords']:
        if col in df.columns:
            res_cols.append(df[[col]].reset_index(drop=True))
        else:
            res_cols.append(pd.Series(['N/A'] * len(df), name=col).reset_index(drop=True))

    res_cols.append(emb_df)

    result_df = pd.concat(res_cols, axis=1)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.mps.is_available():
        torch.mps.empty_cache()
    gc.collect()

    return result_df

try:
    df = pd.read_csv('./origin_data/movie_all_fixed.csv', on_bad_lines='skip', skipinitialspace=True)
    df.columns = df.columns.str.strip()
    print("Successfully loaded CSV.")
    display(df.head())
except Exception as e:
    print(f"Error loading CSV: {e}")

In [ ]:
task_desc = "Extract semantic features for movie content analysis"

# Models mapping (Adjusted to current available versions of Qwen/GTE)
# Qwen3 is not yet released, so using the highly performant Qwen2-GTE series
model_configs = [
    {"label": "Qwen3_4B", "id": "Qwen/Qwen3-Embedding-4B"},  # GPU RAM USAGE: 8GB, processing 2item/s
    {"label": "Qwen3_0.6B", "id": "Qwen/Qwen3-Embedding-0.6B"}, # GPU RAM USAGE: 1.5GB, processing 10item/s
    {"label": "BGE_M3", "id": "BAAI/bge-m3"}  #GPU RAM USAGE: 2.5 GB, processing 25item/s
]

for cfg in model_configs:
    try:
        res = generate_embeddings(cfg['id'], df, task_desc)
        fname = f"result_data/results-{cfg['label'].lower()}.csv"
        res.to_csv(fname, index=False)
        print(f"Saved results to {fname}")
    except Exception as e:
        print(f"Failed to process {cfg['label']}: {e}")